In [1]:
import pandas as pd
import torch
import numpy as np

In [2]:
data = pd.read_csv("../gesture_classification_dataset.csv")
data.columns

Index(['Unnamed: 0', 'hand', 'confidence', 'x0', 'y0', 'z0', 'x1', 'y1', 'z1',
       'x2', 'y2', 'z2', 'x3', 'y3', 'z3', 'x4', 'y4', 'z4', 'x5', 'y5', 'z5',
       'x6', 'y6', 'z6', 'x7', 'y7', 'z7', 'x8', 'y8', 'z8', 'x9', 'y9', 'z9',
       'x10', 'y10', 'z10', 'x11', 'y11', 'z11', 'x12', 'y12', 'z12', 'x13',
       'y13', 'z13', 'x14', 'y14', 'z14', 'x15', 'y15', 'z15', 'x16', 'y16',
       'z16', 'x17', 'y17', 'z17', 'x18', 'y18', 'z18', 'x19', 'y19', 'z19',
       'x20', 'y20', 'z20', 'video', 'id', 'label'],
      dtype='str')

In [3]:
X = data.drop(columns = ["Unnamed: 0", "confidence", "video", "id", "label"] )
Y = data['label']
X.head()

,hand,x0,y0,z0,x1,y1,z1,x2,y2,z2,...,z17,x18,y18,z18,x19,y19,z19,x20,y20,z20
0,Right,0.0,0.0,0.0,0.002357,-0.046531,-0.003313,0.015087,-0.091500,-0.008970,...,-0.017369,0.064948,-0.002125,-0.018853,0.056544,0.002580,-0.015215,0.045729,0.004943,-0.012389
1,Right,0.0,0.0,0.0,0.003563,-0.045035,-0.003620,0.016583,-0.087514,-0.009416,...,-0.016987,0.064005,0.001478,-0.018412,0.055614,0.006124,-0.014664,0.045403,0.008020,-0.011686
2,Right,0.0,0.0,0.0,0.003470,-0.045270,-0.002904,0.015930,-0.085044,-0.008400,...,-0.016100,0.061116,0.003558,-0.016904,0.053715,0.007098,-0.013519,0.044111,0.009054,-0.010989
3,Right,0.0,0.0,0.0,0.003813,-0.044966,-0.003099,0.016591,-0.083982,-0.008629,...,-0.016239,0.060556,0.004051,-0.017317,0.053062,0.007506,-0.013876,0.043321,0.008789,-0.011282
4,Right,0.0,0.0,0.0,0.003478,-0.042344,-0.003210,0.015902,-0.080038,-0.008412,...,-0.016328,0.058304,0.007675,-0.018383,0.050211,0.011911,-0.015015,0.040619,0.013236,-0.012110


In [4]:
def get_landmark(data, idx):
    return data[ [f'x{idx}', f'y{idx}', f'z{idx}'] ].to_numpy()

def get_direction_vectors(data, idx1, idx2):
    vec1 = get_landmark(X, idx1)
    vec2 = get_landmark(X, idx2)

    vec = vec1 - vec2

    vec /= np.linalg.norm(vec, axis=1, keepdims=True)

    return vec 



**Using the following map from mediapipe**

![Landmarks map](landmarks.png)

In [5]:
fingers = {
    'thumb' : (4,2),
    'index' : (8,6),
    'middle' : (12,10),
    'ring' : (16,14),
    'pinky' : (20,18),
    'palm' : (9,0)
}



for name, idx in fingers.items():
    vector = get_direction_vectors(X, idx[0], idx[1])

    X[f'{name}_dx'] = vector[:,0]
    X[f'{name}_dy'] = vector[:,1]
    X[f'{name}_dz'] = vector[:,2]

X.head()

,hand,x0,y0,z0,x1,y1,z1,x2,y2,z2,...,middle_dz,ring_dx,ring_dy,ring_dz,pinky_dx,pinky_dy,pinky_dz,palm_dx,palm_dy,palm_dz
0,Right,0.0,0.0,0.0,0.002357,-0.046531,-0.003313,0.015087,-0.091500,-0.008970,...,0.013942,-0.876091,0.438300,0.200892,-0.895019,0.329136,0.301017,0.516599,-0.838011,-0.175677
1,Right,0.0,0.0,0.0,0.003563,-0.045035,-0.003620,0.016583,-0.087514,-0.009416,...,0.037099,-0.872764,0.434937,0.221615,-0.892867,0.313969,0.322819,0.569433,-0.801273,-0.183596
2,Right,0.0,0.0,0.0,0.003470,-0.045270,-0.002904,0.015930,-0.085044,-0.008400,...,0.047592,-0.882499,0.426415,0.198407,-0.903338,0.291984,0.314206,0.587326,-0.789405,-0.178573
3,Right,0.0,0.0,0.0,0.003813,-0.044966,-0.003099,0.016591,-0.083982,-0.008629,...,0.072097,-0.885146,0.404023,0.230829,-0.913553,0.251170,0.319899,0.587709,-0.786788,-0.188584
4,Right,0.0,0.0,0.0,0.003478,-0.042344,-0.003210,0.015902,-0.080038,-0.008412,...,0.018485,-0.899921,0.383434,0.207653,-0.903614,0.284155,0.320528,0.593047,-0.783756,-0.184452


In [7]:
#encoding the classes
from sklearn.preprocessing import LabelEncoder


label_encoder = LabelEncoder()
hand_encoder = LabelEncoder()

X['hand'] = hand_encoder.fit_transform(X['hand'])

Y = label_encoder.fit_transform(Y)

print(X.head())
print(Y)

   hand   x0   y0   z0        x1        y1        z1        x2        y2  \
0     1  0.0  0.0  0.0  0.002357 -0.046531 -0.003313  0.015087 -0.091500   
1     1  0.0  0.0  0.0  0.003563 -0.045035 -0.003620  0.016583 -0.087514   
2     1  0.0  0.0  0.0  0.003470 -0.045270 -0.002904  0.015930 -0.085044   
3     1  0.0  0.0  0.0  0.003813 -0.044966 -0.003099  0.016591 -0.083982   
4     1  0.0  0.0  0.0  0.003478 -0.042344 -0.003210  0.015902 -0.080038   

         z2  ...  middle_dz   ring_dx   ring_dy   ring_dz  pinky_dx  pinky_dy  \
0 -0.008970  ...   0.013942 -0.876091  0.438300  0.200892 -0.895019  0.329136   
1 -0.009416  ...   0.037099 -0.872764  0.434937  0.221615 -0.892867  0.313969   
2 -0.008400  ...   0.047592 -0.882499  0.426415  0.198407 -0.903338  0.291984   
3 -0.008629  ...   0.072097 -0.885146  0.404023  0.230829 -0.913553  0.251170   
4 -0.008412  ...   0.018485 -0.899921  0.383434  0.207653 -0.903614  0.284155   

   pinky_dz   palm_dx   palm_dy   palm_dz  
0  0.301017 

In [8]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X, Y, train_size=0.2, random_state= 42)

x_train_tensor = torch.tensor(x_train.values, dtype = torch.float32)
x_test_tensor = torch.tensor(x_test.values, dtype = torch.float32)

y_train_tensor = torch.tensor(y_train)
y_test_tensor = torch.tensor(y_test)

x_train.shape

(314, 82)

In [22]:
class GestureClassifier(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.network = torch.nn.Sequential(
            torch.nn.Linear(82, 64),
            torch.nn.ReLU(),

            torch.nn.Linear(64, 32),
            torch.nn.ReLU(),

            torch.nn.Dropout(0.2),

            torch.nn.Linear(32, 5)
        )

    def forward(self, x):
        return self.network(x)

    

In [23]:
model = GestureClassifier()

loss_fn = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

epochs = 1000

In [24]:
for t in range(epochs):

    optimizer.zero_grad()

    y_pred = model(x_train_tensor)

    loss = loss_fn(y_pred, y_train_tensor)

    if (t+1) % 50 == 0:
        print(f"Epoch: {t+1}, Loss: {loss.item()}")

    loss.backward()

    optimizer.step()

Epoch: 50, Loss: 0.8769921660423279
Epoch: 100, Loss: 0.4078505337238312
Epoch: 150, Loss: 0.23873135447502136
Epoch: 200, Loss: 0.17963585257530212
Epoch: 250, Loss: 0.13354437053203583
Epoch: 300, Loss: 0.12400193512439728
Epoch: 350, Loss: 0.11765237152576447
Epoch: 400, Loss: 0.08896760642528534
Epoch: 450, Loss: 0.09344261139631271
Epoch: 500, Loss: 0.08636429160833359
Epoch: 550, Loss: 0.06639192998409271
Epoch: 600, Loss: 0.07080668210983276
Epoch: 650, Loss: 0.06755541265010834
Epoch: 700, Loss: 0.05980855971574783
Epoch: 750, Loss: 0.05626954138278961
Epoch: 800, Loss: 0.05881272628903389
Epoch: 850, Loss: 0.04494976997375488
Epoch: 900, Loss: 0.05334930494427681
Epoch: 950, Loss: 0.04560871049761772
Epoch: 1000, Loss: 0.04096256569027901


In [25]:
model.eval()

with torch.no_grad():
    train_pred = model(x_train_tensor).argmax(dim=1).cpu().numpy()
    test_pred = model(x_test_tensor).argmax(dim=1).cpu().numpy()
test_pred

array([2, 3, 2, ..., 2, 2, 4], shape=(1259,))

In [26]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [27]:
print("Train accuracy: ", accuracy_score(train_pred, y_train))
print("Test accuracy: ", accuracy_score(test_pred, y_test))

Train accuracy:  0.9872611464968153
Test accuracy:  0.9126290706910246


In [28]:
print(confusion_matrix(test_pred, y_test))

[[158   0   1   0   0]
 [  1 148   4  27   2]
 [  5   7 457   2   4]
 [  0  30  25 121   0]
 [  0   1   1   0 265]]


In [29]:
print(classification_report(test_pred, y_test))

              precision    recall  f1-score   support

           0       0.96      0.99      0.98       159
           1       0.80      0.81      0.80       182
           2       0.94      0.96      0.95       475
           3       0.81      0.69      0.74       176
           4       0.98      0.99      0.99       267

    accuracy                           0.91      1259
   macro avg       0.90      0.89      0.89      1259
weighted avg       0.91      0.91      0.91      1259



In [30]:
label_encoder.classes_

array([0, 1, 2, 3, 4])

In [ ]:
## the model this time has worsen the performance. Will need to try something else
import json
import pickle as pk

torch.save(model.state_dict(), "state_dict.pth")

with open("label_encoder.pkl" , "wb") as f:
    pk.dump(label_encoder, f)
    
with open( "hand_encoder.pkl" , "wb") as f:
    pk.dump(hand_encoder, f)

config = {
    
    "num_features": 64,
    "num_classes": 5,
    "class_names": list(label_encoder.classes_)
}

with open( "config.json", "w") as f:
    json.dump(config, f, indent=4)

(1573, 82)

Here i give up and will go with the ver1 itself. Maybe someone else in the future will contribute and 
make this model a great one!